# Bước 5: Đánh giá Sự Lệch pha (Dissonance / Gap Analysis)

Jupyter Notebook này thực hiện **Phần 2.3** nhưng điều chỉnh cho bài toán **Single Case Study (1 Ngân hàng)**. Thay vì đo hệ số tương quan Pearson (yêu cầu số lượng lớn các ngân hàng tham gia cùng lúc), thuật toán sẽ đo lường **Sự chênh lệch góc nhìn (Dissonance Gap - Delta)** giữa Báo cáo tự công bố và Tin tức truyền thông.

### Quy trình tính toán Gap Score (Delta):
1. **Tổng hợp Điểm số:** 
   - **X (Report Score)**: Điểm cảm xúc trích xuất từ 1 Báo cáo của công ty.
   - **Y_mean (Mean News Score)**: Điểm cảm xúc trung bình ($\bar{Y}$) trích xuất từ tất cả các Tin tức báo chí.
2. **Tính Toán Độ lệch (Delta / Gap):** Áp dụng công thức:
   $$\Delta_{Gap} = X - \bar{Y}$$
3. **Đánh giá & Diễn giải (Interpretation):**
   - Nếu $\Delta$ dương lớn: Báo cáo ngợi khen quá mức trong khi thực tế dư luận không thấy vậy $\rightarrow$ Cảnh báo **Greenwashing (Dissonance)**.
   - Nếu $\Delta$ xoay quanh 0: Nhất quán giữa báo cáo và báo chí $\rightarrow$ **Credible communication** (Đáng tin cậy).

In [4]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# =============== CẤU HÌNH ĐƯỜNG DẪN =============== 
# --- CẤU HÌNH NĂM VÀ NGÂN HÀNG CẦN XỬ LÝ ---
TARGET_YEAR = "2023"
TARGET_BANK = "Vietcombank"  # Điền tên ngân hàng (vd: "vietcombank", "bidv", "vietinbank")
# ------------------------------------------

PROJECT_ROOT = Path.cwd()
YEAR = TARGET_YEAR
BANK_NAME = TARGET_BANK

# Đường dẫn folder Scoring tự động ghép nối theo Tên Ngân Hàng
SCORING_DIR = PROJECT_ROOT / "data" / YEAR / BANK_NAME / "scoring"

print(f"Ngân hàng đang xét: {BANK_NAME}")
print(f"Thư mục Scoring: {SCORING_DIR}")

Ngân hàng đang xét: Vietcombank
Thư mục Scoring: d:\NCKH\Thread_1\data\2023\Vietcombank\scoring


In [5]:
# =============== BƯỚC 1: LOAD DỮ LIỆU ĐÃ CHẤM ĐIỂM =============== 
report_scores = []
news_scores = []

# Đọc file điểm Báo cáo (Vector X)
# Vì giờ mỗi file báo cáo tự động lưu tên theo dạng 'scored_BIDV.json' 
# -> Dùng glob để bắt tất cả các file bắt đầu bằng 'scored_' MÀ KHÔNG CÓ chữ 'esg_check_llms' 
report_files = [f for f in SCORING_DIR.glob("scored_*.json") if "esg_check_llms" not in f.name]

for rep_file in report_files:
    with open(rep_file, 'r', encoding='utf-8') as f:
        report_data = json.load(f)
        scores = [item.get('comparative_score', 0) for item in report_data.get('reports', [])]
        if scores:
            report_scores.extend(scores)

# Đọc file điểm Tin tức (Tất cả bài báo làm thành Vector Y)
news_files = list(SCORING_DIR.glob("scored_esg_check_llms_*.json"))
for nf in news_files:
    with open(nf, 'r', encoding='utf-8') as f:
        news_data = json.load(f)
        scores = [item.get('comparative_score', 0) for item in news_data.get('items', [])]
        if scores:
            news_scores.extend(scores)

print(f"-> Lấy được {len(report_scores)} đầu mục điểm từ Báo cáo của {BANK_NAME}.")
print(f"-> Lấy được {len(news_scores)} điểm độc lập từ lượng Tin tức của {BANK_NAME}.\n")

# XỬ LÝ DỮ LIỆU BÀI TOÁN GAP ANALYSIS:
if not report_scores or not news_scores:
    print(f"❌ Không đủ dữ liệu ở 1 trong 2 phía Báo cáo/Tin tức của {BANK_NAME} để tính Gap.")
    report_mean = 0
    news_mean = 0
else:
    report_mean = np.mean(report_scores) 
    news_mean = np.mean(news_scores)

df_analysis = pd.DataFrame({
    "Metric": [f"Báo cáo tự đánh giá ({BANK_NAME})", f"Truyền thông báo chí ({BANK_NAME})"],
    "Mean_Comparative_Score": [report_mean, news_mean],
    "Document_Count": [len(report_scores), len(news_scores)]
})
display(df_analysis)

-> Lấy được 1 đầu mục điểm từ Báo cáo của Vietcombank.
-> Lấy được 35 điểm độc lập từ lượng Tin tức của Vietcombank.



,Metric,Mean_Comparative_Score,Document_Count
0,Báo cáo tự đánh giá (Vietcombank),0.017827,1
1,Truyền thông báo chí (Vietcombank),0.007697,35


In [6]:
# =============== BƯỚC 2: TÍNH GAP SCORE (DELTA) =============== 

delta_gap = report_mean - news_mean

print("===== KẾT QUẢ TÍNH TOÁN GAP SCORE (DISSONANCE) =====")
print(f"Điểm Báo cáo tự công bố  (X) : {report_mean:.6f}")
print(f"Điểm Trung bình Báo chí (Y_mean): {news_mean:.6f}")
print(f"Độ lệch Delta (Gap)          : {delta_gap:.6f}")
print("------------------------------------------------------")

# Lưu kết quả vào file Excel trong folder scoring
gap_results = {
    'Bank_Name': [BANK_NAME],
    'Report_Score_X': [report_mean],
    'News_Score_Y_mean': [news_mean],
    'Delta_Gap': [delta_gap]
}

df_gap = pd.DataFrame(gap_results)
output_excel = SCORING_DIR / f"{BANK_NAME}_Gap_Score.xlsx"
df_gap.to_excel(output_excel, index=False)
print(f"✅ Đã lưu kết quả Gap Score vào file Excel: {output_excel}")


===== KẾT QUẢ TÍNH TOÁN GAP SCORE (DISSONANCE) =====
Điểm Báo cáo tự công bố  (X) : 0.017827
Điểm Trung bình Báo chí (Y_mean): 0.007697
Độ lệch Delta (Gap)          : 0.010129
------------------------------------------------------
✅ Đã lưu kết quả Gap Score vào file Excel: d:\NCKH\Thread_1\data\2023\Vietcombank\scoring\Vietcombank_Gap_Score.xlsx
